In [ ]:
import sys

assert sys.executable.split('/')[5]=='wakai'

In [ ]:
# TIC*1445
target = {'pl_orbper': (3.272052,1e-05), #best['b_period'],
          'pl_rade': (5.7,0.7), #(Rp_samples.median(), Rp_samples.std()),
}
target

In [ ]:
# TOI-6715
target2 = {'pl_orbper': (2.8625896500316537, 3.263991441179825e-06),
 # 'pl_rade': (3.8858759003555825, 0.2871934014216958)
 'pl_rade': (4.75, 0.3)
          }

In [ ]:
# TOI-5973
target3 = {'pl_orbper': (3.16, 1e-05),
 'pl_rade': (8.67, 0.1)}

In [ ]:
import astropy.units as u

# TOI-5973
target4 = {'pl_orbper': (3.16, 1e-05),
 'pl_rade': (2.17026355*u.Rjup.to(u.Rearth),
             2.5e-7*u.Rjup.to(u.Rearth))
          }

In [ ]:
# TOI-6940
target5 = {'pl_orbper': (0.3052608, 5e-7),
 'pl_rade': #(2.03448, 0.377961)
           (1.4, 0.17)
          }

In [ ]:
# TOI-5169
target6 = {'pl_orbper': (8.0387972, 0.0000287),
 'pl_rade': (2.25604, 0.174709)
          }

In [ ]:
# TOI-5734
target7 = {'pl_orbper': (6.1841876, 0.0000080),
 'pl_rade': (2.1, 0.12)
          }

In [ ]:
special = [#'toi-5734', #Morello et al. 2026
           'hip67522', #Barber et al. 2024
           'tic434398831' #Vach et al. 2025
          ]

## neptune desert

In [ ]:
import sys

sys.path.insert(0, '/ut2/jerome/github/research/project/TOI-6715')

from toi6715 import load_plot_params

plot_params = load_plot_params()

In [ ]:
from toi6715 import get_nexsci_data

table_name = 'pscomppars'
df_nexsci = get_nexsci_data(table_name)

In [ ]:
# get values from default references
if table_name=="ps":
    idx = df_nexsci['default_flag'] == 1
    all_hostnames = df_nexsci[idx].hostname.unique()
    
    #TOI-1136
    idx = df_nexsci[df_nexsci.hostname=="TOI-1136"][df_nexsci[df_nexsci.hostname=="TOI-1136"].pl_refname.str.contains("Dai")].index
    df_nexsci.loc[idx,'default_flag']=1
else:
    all_hostnames = df_nexsci.hostname.unique()
len(all_hostnames)

In [ ]:
import astropy.units as u

# get only transiting planets
df_nexsci['pl_rade'] = df_nexsci['pl_radj']*u.Rjup.to(u.Rearth)

# filter data
idx = (df_nexsci.pl_orbper>0) & \
        (df_nexsci.pl_rade>0) & \
        (df_nexsci.st_mass>0) & \
        (df_nexsci.discoverymethod.isin(['Transit',
                                         # 'Radial Velocity' #To include hd114082b: Kakhozai+2024
                                        ])) #& \
        # (df_nexsci['default_flag'] == 1)
df = df_nexsci[idx]
df.shape

In [ ]:
import pandas as pd

df_hosts = pd.read_csv("/ut2/jerome/github/research/project/TOI-6715/data/young_validated_planets_TOIs - validated_hosts.csv", 
                        skiprows=[1], comment='#').dropna(subset=['Host']).reset_index(drop=True)
# df_hosts = df_hosts[df_hosts.distance<500].reset_index(drop=True)
df_hosts.shape

In [ ]:
idx = df_hosts.query("Host=='Kepler-289'").index[0]
idx

In [ ]:
df_hosts = df_hosts.iloc[:idx]
df_hosts.shape

In [ ]:
i = df_hosts.query("Host=='KOI-7913'").index[0]
df_hosts.at[i,'Host'] = 'KOI-7913A'

In [ ]:
df_hosts['hostname'] = df_hosts.Host.apply(lambda x: x.lower().replace(' ', ''))
young_hostnames = df_hosts['hostname'].reset_index(drop=True)
young_hostnames

In [ ]:
# add a column for edited hostname
df['hostname'] = df.hostname.apply(lambda x: x.lower().replace(' ', ''))

In [ ]:
# cross-match NExSci data with own my list based on (edited) hostname
idx = df['hostname'].isin(young_hostnames)
df_planets = df[idx]
df_planets.shape

In [ ]:
#planets in my list not found in NExSci
found=df['hostname'][idx].tolist()
young_hostnames[~young_hostnames.isin(found)]

In [ ]:
df_hosts['Age_lower'] = df_hosts['Age'].apply(lambda x: str(x).split('-')[0]).astype(float).values
df_hosts['Age_upper'] = df_hosts['Age'].apply(lambda x: str(x).split('-')[1]).astype(float).values

In [ ]:
def theoretical_boundary_loglog(period, method='stripe'):
    """
    neptune desert boundary in log space
    """
    if method=='stripe':
        lower = 0.68*period
        upper = -0.33*period+1.17
    elif method=='likelihood':
        from uncertainties import ufloat
        a=ufloat(0.67,0.06)
        b=ufloat(0.01,0.04)
        lower = a*period-b
        
        a=ufloat(0.31,0.12)
        b=ufloat(1.19,0.06)
        upper = -a*period+b
    return lower, upper

def theoretical_boundary(period):
    """
    neptune desert boundary in linear space
    """
    lower = period**0.68
    upper = 14.8*period**-0.33
    return lower, upper

def vaneylen_model_linear(p): 
    """
    period-radius
    
    https://ui.adsabs.harvard.edu/abs/2018MNRAS.479.4786V/abstract
    """
    y1 = p**(-0.09+0.02)*10**(0.37+0.04)
    y2 = p**(-0.09-0.04)*10**(0.37-0.02)
    y = p**-0.09*(10**0.37)
    return y

def vaneylen_model_loglog(x): 
    """
    period-radius: log10R = mlog10P + a
    
    https://ui.adsabs.harvard.edu/abs/2018MNRAS.479.4786V/abstract
    """
    y1 = x*(-0.09+0.02)+(0.37+0.04)
    y2 = x*(-0.09-0.04)+(0.37-0.02)
    return y1, y2

#@|-----------------------------------------------------------------
#@|Boundaries of the Neptunian desert (Castro-González et al. 2024)
#@|-----------------------------------------------------------------

def empirical_boundary_loglog():
    x_tw_low = np.linspace(-0.30, np.log10(2.97), 100)
    x_tw_up = np.linspace(np.log10(2.97),0.120,  100)
    y_tw_up = -0.43 * x_tw_up + 1.14
    y_tw_low = +0.55 * x_tw_low + 0.36
    
    # print(x_tw_low,y_tw_low)
    # print(x_tw_up,y_tw_up)
    
    x1 = np.tile(-0.30,100)
    y1= np.linspace(-0.07,0.191,100)
    
    x2 = np.tile(0.12,100)
    y2 = np.linspace(1.0884,1.20,100)

    x  = np.concatenate([x1,x_tw_low,x_tw_up,x2])
    y = np.concatenate([y1,y_tw_low,y_tw_up,y2])
    return x,y

def empirical_boundary_linear():
    # Generate log10-scaled x values
    x_tw_low_log = np.linspace(-0.30, np.log10(2.97), 100)
    x_tw_up_log = np.linspace(np.log10(2.97), 0.120, 100)

    # Corresponding log10-scaled y values
    y_tw_up_log = -0.43 * x_tw_up_log + 1.14
    y_tw_low_log = +0.55 * x_tw_low_log + 0.36

    # Vertical boundaries in log space
    x1_log = np.tile(-0.30, 100)
    y1_log = np.linspace(-0.07, 0.191, 100)

    x2_log = np.tile(0.12, 100)
    y2_log = np.linspace(1.0884, 1.20, 100)

    # Combine all log-space data
    x_log = np.concatenate([x1_log, x_tw_low_log, x_tw_up_log, x2_log])
    y_log = np.concatenate([y1_log, y_tw_low_log, y_tw_up_log, y2_log])

    # Convert to linear space
    x_linear = 10 ** x_log
    y_linear = 10 ** y_log

    return x_linear, y_linear

In [ ]:
import re
import seaborn as sb
import numpy as np
import matplotlib.pyplot as pl
import matplotlib.ticker as mticker
from collections import OrderedDict
from matplotlib.patches import Rectangle
pl.rcParams['font.size'] = 14

#####
x = 'pl_orbper'
y = 'pl_rade'
show_target = True
show_contour = True
show_young_planets = True
show_empirical_boundary = False
show_theoretical_boundary = True
show_period_radius_gap = False
porb_lims = (0.2, 300)
rp_lims = (0.6, 35)
figsize = (5, 5)
legend = ' Our Work'
#####

fig, ax = pl.subplots(figsize=figsize)

# contour plot of all planets
if show_contour:
    _ = sb.kdeplot(data=df, 
                   x=x, 
                   y=y,  
                   cmap='gray_r', 
                   # n_levels=8, 
                   log_scale=True, 
                   fill=True, 
                   ax=ax, 
                   zorder=-1
                  )
    
# scatter plot of young planets
if show_young_planets:
    periods = df_planets['pl_orbper']
    radii = df_planets['pl_rade']
    names = df_planets['hostname']
    
    # color depending on age
    for xx,yy,n in zip(periods, radii, names):
        idx = young_hostnames==n
        d = df_hosts[idx].squeeze()
        d = d.iloc[0] if isinstance(d, pd.DataFrame) else d #multiple rows
        if len(d)>0:
            try:
                age = float(str(d.Age).split('-')[1])
                marker = '*' if n in special else 'o'
                ec = 'w' #'k' if n in special else 'w'
                s = 14 if n in special else 9
                zorder = 100 if n in special else 10
                if marker=='*':
                    marker = 'o' if n=='hip67522' and int(xx)==14 else '*'
                    s = 9 if n=='hip67522' and int(xx)==14 else 14
                    # if marker=='*':
                    #     print(f"{n}: P={xx},R={yy}")
                if age<100:
                    print(f"{n}: P={xx},R={yy}")
                    color = 'C3'
                    lbl = 'age<100Myr'
                    # zorder = 12
                    ax.plot(xx, yy, marker, label=lbl, ms=s, mec=ec, c=color, zorder=zorder)
                     # white outline
                    # ax.plot(xx, yy, 'o', c='w', label='_nolegend_', ms=8, zorder=6)
                else:
                    # if marker=='*':
                    #     print(f"{n}: P={xx},R={yy}")
                    if n=='toi-5174':
                        print(f"{n}: P={xx},R={yy}")
                    color = 'C0'
                    zorder -= 1
                    lbl = '100Myr<age<1Gyr'
                    ax.plot(xx, yy, marker, ms=s, label=lbl, mec=ec, c=color, zorder=zorder)
                     # white outline
                    # ax.plot(xx, yy, 'o', c='k', label='_nolegend_', ms=8, zorder=6)
                
            except Exception as e:
                 print(f"Error: {n}\n{e}")

if show_target:
    ec = 'w'
    ax.scatter(target3[x], target3[y], marker='*', s=200, color='C3', edgecolors=ec, label=legend, 
               zorder=10)
    ax.scatter(target[x], target[y], marker='*', s=200, color='C3', edgecolors=ec, #label='TIC*1445b', 
               zorder=10)
    ax.scatter(target2[x], target2[y], marker='*', s=200, color='C3', edgecolors=ec, #label='TOI-6715b', 
               zorder=10)
    ax.scatter(target4[x], target4[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-1355', 
               zorder=10)
    ax.scatter(target5[x], target5[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-6940b', 
               zorder=10)
    ax.scatter(target6[x], target6[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-5169b', 
               zorder=10)
    ax.scatter(target7[x], target7[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-5734', 
               zorder=10)
        
# edit axes
ax.set_xlim(*porb_lims)
ax.set_ylim(*rp_lims)
ax.set_ylabel(r'Planet radius (R$_{\oplus}$)')
ax.set_xlabel('Orbital Period (days)')
ax.tick_params(direction='in', which='both', width=2, bottom=True, top=True, left=True, right=True)
lw=2
ax.spines['left'].set_linewidth(lw)
ax.spines['right'].set_linewidth(lw)
ax.spines['top'].set_linewidth(lw)
ax.spines['bottom'].set_linewidth(lw)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

# Neptune desert
if show_theoretical_boundary:
    xs = np.linspace(0.5, 14, 100)
    ys_lo, ys_hi = theoretical_boundary(xs)
    ax.plot(xs, ys_lo,'k',ls='--',lw=2,zorder=1,alpha=0.5)
    ax.plot(xs, ys_hi,'k',ls='--',lw=2,zorder=1,alpha=0.5)

if show_empirical_boundary:
    xbounds, ybounds = empirical_boundary_linear()
    ax.plot(xbounds, ybounds, c='k', linestyle="-.", lw=2, zorder=0, alpha=0.5)

# period-radius gap
if show_period_radius_gap:
    xmodel = np.linspace(0.1, 100, 101)
    ymodel = vaneylen_model_linear(xmodel)
    ax.plot(xmodel, ymodel, 'k', ls=':')

# add legend
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

custom_handles = [
    Line2D([0], [0], marker='o', color='C3', ms=8, mec='w', ls='none', label='age<100Myr'),
    Line2D([0], [0], marker='o', color='C0', ms=8, mec='w', ls='none', label='100Myr<age<1Gyr'),
    (Line2D([0], [0], marker='*', color='C3', ms=12, mec=ec, ls='none'),
     Line2D([0], [0], marker='*', color='C0', ms=12, mec=ec, ls='none')),
]

custom_labels = ['age<100Myr', '100Myr<age<1Gyr', legend]

leg = ax.legend(custom_handles, custom_labels,
                handler_map={(tuple): HandlerTuple(ndivide=None)},
                fontsize=12,
                framealpha=1,
                loc='upper right',
                markerfirst=True,
                handletextpad=0.1,
                edgecolor='k')
leg.get_frame().set_facecolor('w')
leg.get_frame().set_facecolor('w')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
fig.savefig('../figures/neptune_desert.png', dpi=200, bbox_inches='tight')

In [ ]:
# fp = '../figures/Kiban-S.png'
# fig.savefig(fp, bbox_inches='tight', dpi=150)
fig.savefig("context.pdf", bbox_inches='tight', dpi=200)

In [ ]:
import re
import seaborn as sb
import numpy as np
import matplotlib.pyplot as pl
import matplotlib.ticker as mticker
from collections import OrderedDict
from matplotlib.patches import Rectangle
pl.rcParams['font.size'] = 14

#####
x = 'pl_orbper'
y = 'pl_rade'
show_target = True
show_contour = True
show_young_planets = True
show_empirical_boundary = False
show_theoretical_boundary = True
show_period_radius_gap = False
porb_lims = (1, 100)
rp_lims = (1.5, 25)
figsize = (5, 5)
# legend = 'Kiban-S'
legend = 'Our Work'
#####

fig, ax = pl.subplots(figsize=figsize)

# contour plot of all planets
if show_contour:
    _ = sb.kdeplot(data=df, 
                   x=x, 
                   y=y,  
                   cmap='gray_r', 
                   # n_levels=8, 
                   log_scale=True, 
                   fill=True, 
                   ax=ax, 
                   zorder=-1
                  )
    
# scatter plot of young planets
if show_young_planets:
    periods = df_planets['pl_orbper']
    radii = df_planets['pl_rade']
    names = df_planets['hostname']
    
    # color depending on age
    for xx,yy,n in zip(periods, radii, names):
        idx = young_hostnames==n
        d = df_hosts[idx].squeeze()
        d = d.iloc[0] if isinstance(d, pd.DataFrame) else d #multiple rows
        if len(d)>0:
            try:
                age = float(str(d.Age).split('-')[1])
                marker = '*' if n in special else 'o'
                s = 14 if n in special else 9
                zorder = 100 if n in special else 10
                if marker=='*':
                    marker = 'o' if n=='hip67522' and int(xx)==14 else '*'
                    if n=='hip67522' and int(xx)==14:
                        continue
                if age<100:
                    # print(f"{n}: P={xx},R={yy}")
                    color = 'C3'
                    lbl = 'age<100Myr'
                    # zorder = 12
                    ax.plot(xx, yy, marker, label=lbl, ms=s, mec='w', c=color, zorder=zorder)
                     # white outline
                    # ax.plot(xx, yy, 'o', c='w', label='_nolegend_', ms=8, zorder=6)
                else:
                    continue
                
            except Exception as e:
                 print(f"Error: {n}\n{e}")

if show_target:
    ax.scatter(target3[x], target3[y], marker='*', s=200, color='C3', edgecolors='w', label=legend, 
               zorder=10)
    ax.scatter(target[x], target[y], marker='*', s=200, color='C3', edgecolors='w', #label='TIC*1445b', 
               zorder=10)
    ax.scatter(target2[x], target2[y], marker='*', s=200, color='C3', edgecolors='w', #label='TOI-6715b', 
               zorder=10)
        
# edit axes
ax.set_xlim(*porb_lims)
ax.set_ylim(*rp_lims)
ax.set_ylabel(r'Planet radius (R$_{\oplus}$)')
ax.set_xlabel('Orbital Period (days)')
ax.tick_params(direction='in', which='both', width=2, bottom=True, top=True, left=True, right=True)
lw=2
ax.spines['left'].set_linewidth(lw)
ax.spines['right'].set_linewidth(lw)
ax.spines['top'].set_linewidth(lw)
ax.spines['bottom'].set_linewidth(lw)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

# Neptune desert
if show_theoretical_boundary:
    xs = np.linspace(0.5, 14, 100)
    ys_lo, ys_hi = theoretical_boundary(xs)
    ax.plot(xs, ys_lo,'k',ls='--',lw=2,zorder=1,alpha=0.5)
    ax.plot(xs, ys_hi,'k',ls='--',lw=2,zorder=1,alpha=0.5)

# Neptunian ridge label
# ax.text(np.sqrt(3.2 * 5.7)-1, 6.5, 'Neptunian\nRidge', 
#         fontsize=14, 
#         # fontstyle='italic',
#         color='k', 
#         alpha=1,
#         ha='left', 
#         zorder=20)

if show_empirical_boundary:
    xbounds, ybounds = empirical_boundary_linear()
    ax.plot(xbounds, ybounds, c='k', linestyle="-.", lw=2, zorder=0, alpha=0.5)

# period-radius gap
if show_period_radius_gap:
    xmodel = np.linspace(0.1, 100, 101)
    ymodel = vaneylen_model_linear(xmodel)
    ax.plot(xmodel, ymodel, 'k', ls=':')

# add legend
handles, labels = pl.gca().get_legend_handles_labels()
by_label = OrderedDict(zip(labels, handles))
leg = pl.legend(by_label.values(), 
                list(by_label.keys()), 
                fontsize=16,
                framealpha=1,
                loc='upper right',
                markerfirst=True,
                handletextpad=0.1,
                #frame_alpha=1,
                edgecolor='k',
          #bbox_to_anchor=(1.6, 0.7), 
         )
# leg.get_frame().set_alpha(0)
leg.get_frame().set_facecolor('w')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))